# Voice-V2 — XTTS v2 fine-tune on Kaggle

**Why Kaggle over Colab:** the **Save Version → Save & Run All (Commit)** button runs this entire notebook on Kaggle's servers in the background. Disconnects don't kill it. When the run finishes, all files in `/kaggle/working/` are saved as the notebook's output and stay there forever.

## Setup checklist
1. **Create a Kaggle account** (kaggle.com — sign in with Google works).
2. **Upload your dataset as a Kaggle Dataset:**
   - Go to kaggle.com/datasets → New Dataset
   - Upload `dataset.zip` (the 21 MB zip from your project root — contains `data/dataset/wavs/*.wav` + `metadata.csv`)
   - Title it something like `voice-v2-dataset` (private is fine)
3. **Create a new Notebook** at kaggle.com/code → New Notebook.
4. In the notebook sidebar (right side), click **+ Add Data** and attach your `voice-v2-dataset` dataset. It will mount at `/kaggle/input/voice-v2-dataset/`.
5. In the sidebar, enable **Accelerator → GPU T4 x2** (or P100 — both work).
6. Paste these cells into the notebook (or use **File → Import Notebook** and pull this `.ipynb` from the GitHub repo).
7. **Click Save Version → Save & Run All (Commit)**. Walk away. Come back when it's done.

In [ ]:
# 1. Inspect the environment + the attached dataset
import os, glob
!nvidia-smi | head -20

# Show whatever's under /kaggle/input/ so we can see what Kaggle mounted
print('\n=== /kaggle/input/ contents ===')
for root, dirs, files in os.walk('/kaggle/input'):
    depth = root[len('/kaggle/input'):].count('/')
    if depth <= 3:
        print(root, '->', sorted(dirs)[:5], '|', sorted(files)[:5])

# Find any directory containing both wavs/ and metadata.csv, recursively
DATASET_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'wavs' in dirs and 'metadata.csv' in files:
        DATASET_PATH = root
        break

if DATASET_PATH is None:
    raise FileNotFoundError(
        'No dataset with wavs/ + metadata.csv found under /kaggle/input/. '
        'Make sure the dataset is attached via the sidebar (+ Add Data).'
    )

OUTPUT_PATH = '/kaggle/working/run'  # persisted with the notebook commit
os.makedirs(OUTPUT_PATH, exist_ok=True)
print('\nDataset:', DATASET_PATH)
print('Output:', OUTPUT_PATH)
print('Clips:', len(os.listdir(f'{DATASET_PATH}/wavs')))

In [ ]:
# 2. Clone the training script from GitHub
%cd /kaggle/working
!rm -rf Voice-V2
!git clone https://github.com/sforslime/Voice-V2.git
%cd /kaggle/working/Voice-V2

In [ ]:
# 3. Install Coqui TTS + pinned transformers (~3 min)
!pip install -q 'coqui-tts[codec]' 'transformers<5'

In [ ]:
# 4. Train. EPOCHS=20 ≈ 740 steps on this dataset → ~15–30 min on T4/P100.
import os
os.environ['DATASET_PATH'] = DATASET_PATH
os.environ['OUTPUT_PATH'] = OUTPUT_PATH
os.environ['BATCH_SIZE'] = '3'
os.environ['GRAD_ACCUM'] = '16'
os.environ['EPOCHS'] = '20'
os.environ['SAVE_STEP'] = '500'
os.environ['COQUI_TOS_AGREED'] = '1'
# Kaggle's T4 x2 exposes 2 GPUs; pin to the first so the trainer doesn't
# refuse to pick one (or switch to `python -m trainer.distribute` for multi-GPU).
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

!python scripts/finetune_xtts.py

In [ ]:
# 5. Stage the final model files in /kaggle/working/finetuned/ so they're easy to download
import glob, shutil, os
FINAL_DIR = '/kaggle/working/finetuned'
os.makedirs(FINAL_DIR, exist_ok=True)

runs = sorted(glob.glob(f'{OUTPUT_PATH}/voice_v2_xtts_ft-*'))
run_dir = runs[-1]
ckpt_path = sorted(glob.glob(f'{run_dir}/best_model*.pth'))[-1]

shutil.copy(ckpt_path, f'{FINAL_DIR}/best_model.pth')
shutil.copy(f'{run_dir}/config.json', f'{FINAL_DIR}/config.json')
shutil.copy(f'{OUTPUT_PATH}/xtts_v2_base/vocab.json', f'{FINAL_DIR}/vocab.json')

print('Final model staged at', FINAL_DIR)
!ls -lh /kaggle/working/finetuned/

In [ ]:
# 6. Quick sanity-check: generate one sample from the fine-tuned model
import glob, os, torch
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
import soundfile as sf
from IPython.display import Audio

config = XttsConfig()
config.load_json(f'{FINAL_DIR}/config.json')
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_path=f'{FINAL_DIR}/best_model.pth',
                     vocab_path=f'{FINAL_DIR}/vocab.json', use_deepspeed=False)
model.cuda()

ref_wav = sorted(glob.glob(f'{DATASET_PATH}/wavs/*.wav'))[0]
out = model.synthesize(
    'Hello, this is my fine-tuned voice. Does this sound more like me now?',
    config, speaker_wav=ref_wav, gpt_cond_len=3, language='en')

OUT_WAV = '/kaggle/working/finetuned_sample.wav'
sf.write(OUT_WAV, out['wav'], 24000)
Audio(OUT_WAV)

## After the commit run finishes

1. Go to your notebook's page on Kaggle.
2. Open the **Output** tab (right sidebar of the notebook page).
3. Download `finetuned/best_model.pth`, `finetuned/config.json`, `finetuned/vocab.json`.
4. On your Mac, drop them into `Voice-V2/models/finetuned/`.
5. Run `.venv/bin/python scripts/infer_finetuned.py` to generate samples locally.